# 07 · R1 — Qwen2-VL-7B zero-shot trên ViTextVQA, đối chiếu Table 7 của paper (W7)

**Vấn đề:** nhóm hỏi "Qwen2.5-VL lệch paper, có nên lấy lại Qwen2?". Con số duy nhất của họ Qwen trên ViTextVQA là **Table 7** của paper ViTextVQA (Nguyen et al., *Expert Systems with Applications* 2025, arXiv:2404.10652v5): dòng **"QwenVL-7b", zero-shot, EM 28,00 · F1 53,80**. Có ba giới hạn cần biết trước khi chạy:

1. Paper chỉ chấm trên **100 câu test chọn ngẫu nhiên** (không công bố là câu nào). Vì vậy bản thân con số 28,00 có sai số khoảng ±9 điểm (CI95).
2. Paper **không công bố prompt, checkpoint và cách chuẩn hoá** khi tính EM/F1. Phần thân bài trích dẫn Qwen2-VL, nên chọn **Qwen2-VL-7B-Instruct**, bản gần nhất còn dùng được với thư viện hiện tại.
3. **Baseline của đề tài vẫn là Qwen2.5-VL-3B** (GVHD đã xác nhận 07/10). R1 **không thay** baseline. R1 chỉ trả lời câu hỏi của nhóm, và R1b là một dòng của ablation "backbone khác nhau" mà đề cương yêu cầu.

| Biến thể | Cấu hình | Mục đích |
|---|---|---|
| **R1** (`qwen2vl7b_paper`) | Qwen2-VL-7B-Instruct, bf16, **không** lượng tử, độ phân giải mặc định của processor | Gần cấu hình paper nhất có thể. So với Table 7 |
| **R1b** (`qwen2vl7b_b1cfg`) | Qwen2-VL-7B-Instruct, **đúng cấu hình B1**: 4-bit NF4, 512 token ảnh | Ablation backbone **3B vs 7B trong điều kiện y hệt** (so ghép cặp với B1) |

Cả hai chạy trên **cùng tập con test 2.000 câu (seed 42) của B1/B2**, cùng prompt zero-shot P0 của B1, greedy, `max_new_tokens` = 32, seed 42, L4.

**Luật (chốt trước khi chạy):**
- **R1 vs Table 7:** NHẤT QUÁN nếu |z| ≤ 1,96 cho **cả EM và F1**. Ở đây z = (nhóm − paper) / √(s²/100 + s²/2000), với s là độ lệch chuẩn từng câu của run nhóm (script `src/eval/paper_consistency.py`).
  - "Nhất quán" **không** có nghĩa là "giống hệt". Với 100 câu, chỉ phát hiện được độ lệch khoảng 8–10 điểm trở lên. Lưu ý: **B1 (Qwen2.5-VL-3B) cũng đã nhất quán với Table 7** (EM 28,80 so với 28,00), nên Table 7 không đủ độ chính xác để phân biệt 3B với 7B.
- **R1b vs B1 (ablation backbone):** khác biệt có ý nghĩa nếu McNemar exact p < 0,05 (EM). ANLS kèm CI95 bootstrap ghép cặp. Kết quả được ghi lại dù 7B tốt hơn hay không.
- Nếu một ảnh làm tràn VRAM ở độ phân giải mặc định (R1), script **tự thu ảnh đó về ≤ 1.280 token**, sinh lại và **ghi tên ảnh vào `oom_fallbacks.json`** để báo cáo.

**Thời gian ước tính:** tải model khoảng 16,6 GB (≈5 phút). R1 ≈35–60 phút, R1b ≈40–50 phút. Tổng ≈1,5–2 giờ, ≈3–4 CU. Một GPU một việc: chạy khi không có run nào khác.

**Nếu Colab rớt:** chạy lại từ phần 1 rồi **Run all**. Các lô đã xong tự bỏ qua.

## 1. Thiết lập môi trường

Giống notebook 02/04/06 (cùng pin `transformers==5.17.0`). Dừng hẳn nếu GPU không phải L4.

*Kỳ vọng: commit mới nhất (≥ commit thêm `--variant`), `GPU: NVIDIA L4`, `✅ Thiết lập xong`.*

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, sys
if not os.path.exists('/content/ViVQA-VLM/.git'):
    !cd /content && git clone -q https://github.com/PLHGNAOH/ViVQA-VLM.git
%cd /content/ViVQA-VLM
!git pull -q origin main
!git log -1 --oneline
sys.path.insert(0, '/content/ViVQA-VLM')

!pip install -q transformers==5.17.0 accelerate qwen-vl-utils bitsandbytes peft huggingface_hub pyyaml

try:
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # không in token ra
    print('HF_TOKEN: đã nạp')
except Exception:
    print('HF_TOKEN: không có (Qwen2-VL-7B-Instruct công khai, vẫn tải được)')

import torch
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU'
print('GPU:', GPU)
if 'L4' not in GPU:
    raise SystemExit('❌ R1 phải chạy trên L4 (cùng GPU với B1/B2). Runtime -> Change runtime type -> L4, rồi chạy lại cell này.')
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print('✅ Thiết lập xong')

## 2. Tên run + kiểm tra trước khi tốn GPU

Kiểm trong vài giây, trước khi tải model: config gốc đúng bản chốt W07, có kết quả B1 2.000 câu để so, và selftest của hai script đạt trên đúng commit này.

*Kỳ vọng: `bfloat16 | 401408 | nf4 | bfloat16`, `B1 test2000: 2000 dự đoán`, Table 7 của B1 = `NHẤT QUÁN…`, hai dòng `SELFTEST OK`.*

In [ ]:
import json, yaml
from src.eval import paper_consistency as pc

DRIVE_ROOT = '/content/drive/MyDrive/ViVQA-VLM'
EXP_ROOT   = f'{DRIVE_ROOT}/experiments'
CHUNK      = 500
RUN_R1B    = True     # False: chỉ chạy R1 (tiết kiệm ≈45 phút)

B1_RUN = 'W07_B1_zeroshot_qwen25vl_vitextvqa_test2000_bf16_L4_seed42'
RUNS = {'R1':  {'variant': 'qwen2vl7b_paper', 'run': 'W07_R1_vitextvqa_test2000_qwen2vl7b_paper_bf16_L4_seed42'},
        'R1b': {'variant': 'qwen2vl7b_b1cfg', 'run': 'W07_R1b_vitextvqa_test2000_qwen2vl7b_b1cfg_L4_seed42'}}
if not RUN_R1B:
    RUNS.pop('R1b')
OUT_DIR = f'{EXP_ROOT}/W07_R1_summary_L4_seed42'

cfg = yaml.safe_load(open('configs/qwen_lora.yaml', encoding='utf-8'))
mdl, q = cfg['model'], cfg['quantization']
print('torch_dtype | max_pixels | quant | compute dtype')
print(f"{mdl['torch_dtype']} | {mdl['max_pixels']} | {q.get('bnb_4bit_quant_type')} | {q.get('bnb_4bit_compute_dtype')}")
assert (mdl['torch_dtype'], mdl['max_pixels'], q.get('bnb_4bit_quant_type'), q.get('bnb_4bit_compute_dtype')) \
       == ('bfloat16', 401408, 'nf4', 'bfloat16'), 'Config KHÔNG phải bản chốt W07 -> dừng, báo Claude'

b1 = json.load(open(f'{EXP_ROOT}/{B1_RUN}/predictions.json', encoding='utf-8'))
print('B1 test2000:', len(b1), 'dự đoán')
assert len(b1) == 2000, 'Thiếu kết quả B1 trên Drive -> chạy notebook 02 trước'
print('B1 so với Table 7 (tham khảo):', pc.table7_report(b1)['verdict'])
for k, r in RUNS.items():
    print(f"  {k:4s} -> {r['run']}  (--variant {r['variant']})")

!python -m src.eval.run_baseline --selftest 2>&1 | tail -n 1
!python -m src.eval.paper_consistency --selftest 2>&1 | tail -n 1

## 3. Chạy R1 rồi R1b (≈1,5–2 giờ)

Mỗi run chạy bằng `!python` (tiến trình riêng) để VRAM được trả lại sạch giữa hai biến thể. Run nào đã có `metrics.json` thì bỏ qua. Lần đầu sẽ tải Qwen2-VL-7B-Instruct (≈16,6 GB) về đĩa Colab.

*Kỳ vọng: dòng `model=Qwen/Qwen2-VL-7B-Instruct | variant=…`, 4 lô mỗi run, mỗi lô in `EM lô = …`, cuối mỗi run `== KẾT QUẢ: EM=…`. R1 có thể in vài dòng `⚠ tràn VRAM ở … -> chạy lại ở …`: bình thường, đã ghi log. Nếu báo lỗi ngay khi nạp model, chụp màn hình gửi Claude.*

In [ ]:
for k, r in RUNS.items():
    run_name, variant = r['run'], r['variant']          # biến đơn để IPython thay vào lệnh !python
    if os.path.exists(f'{EXP_ROOT}/{run_name}/metrics.json'):
        print(f'⏭ {k}: đã có metrics.json -> bỏ qua'); continue
    print(f'\n▶ {k}: {run_name}')
    !python -u -m src.eval.run_baseline --dataset vitextvqa_official --run_name {run_name} --chunk {CHUNK} --variant {variant}

## 4. Kết quả

Ba phần: (1) bảng điểm B1 / R1 / R1b; (2) R1 so với Table 7 theo luật; (3) so ghép cặp R1b vs B1 (ablation backbone, điều kiện y hệt) và R1 vs R1b (cái giá của cấu hình B1 trên 7B).

*Kỳ vọng: một dòng `➜ KẾT LUẬN R1 vs Table 7`, một dòng `➜ KẾT LUẬN ablation backbone`.*

In [ ]:
import re
import pandas as pd

def load(run):
    d = f'{EXP_ROOT}/{run}'
    return (json.load(open(f'{d}/metrics.json', encoding='utf-8')),
            json.load(open(f'{d}/predictions.json', encoding='utf-8')))

LABELS = {'B1': ('B1 · Qwen2.5-VL-3B · 4-bit · 512 token', B1_RUN)}
for k, r in RUNS.items():
    if os.path.exists(f"{EXP_ROOT}/{r['run']}/metrics.json"):
        LABELS[k] = ({'R1': 'R1 · Qwen2-VL-7B · bf16 · độ phân giải mặc định',
                      'R1b': 'R1b · Qwen2-VL-7B · 4-bit · 512 token'}[k], r['run'])
M, P, rows = {}, {}, []
for k, (lab, run) in LABELS.items():
    M[k], P[k] = load(run)
    mm = M[k]
    rows.append({'run': lab, 'EM': 100 * mm['metrics']['exact_match'], 'F1 (phụ)': 100 * mm['metrics'].get('token_f1', float('nan')),
                 'ANLS': 100 * mm['metrics']['anls'], 'VQA-Acc': 100 * mm['metrics']['vqa_accuracy'],
                 's/câu': mm['meta']['sec_per_sample'], 'VRAM đỉnh (GB)': mm['meta'].get('peak_vram_gb_this_session'),
                 'ảnh thu nhỏ do tràn VRAM': len(mm['meta'].get('oom_fallback_images') or []),
                 'câu rỗng': sum(not re.sub(r'[!\s]', '', p['prediction'] or '') for p in P[k])})
rows.append({'run': 'Paper Table 7 · "QwenVL-7b" (100 câu)', 'EM': pc.TABLE7_QWENVL7B['em'], 'F1 (phụ)': pc.TABLE7_QWENVL7B['f1']})
print('==== Bảng 1. ViTextVQA test con 2.000 câu (seed 42), zero-shot ====')
display(pd.DataFrame(rows).round(2))

T7 = {k: pc.table7_report(P[k]) for k in P}
print('\n==== Bảng 2. Đối chiếu Table 7 (luật chốt trước) ====')
display(pd.DataFrame([{'run': k, 'EM nhóm': t['EM']['ours_pct'], 'z(EM)': t['EM']['z'],
                       'F1 nhóm': t['F1']['ours_pct'], 'z(F1)': t['F1']['z'],
                       'lệch nhỏ nhất phát hiện được (EM, điểm)': t['EM']['min_detectable_diff_points'],
                       'kết luận': t['verdict']} for k, t in T7.items()]))
if 'R1' in T7:
    print('➜ KẾT LUẬN R1 vs Table 7:', T7['R1']['verdict'])

CMP = {}
pairs = [('B1', 'R1b', 'ablation backbone 3B vs 7B (điều kiện y hệt)'), ('B1', 'R1', 'B1 vs 7B cấu hình paper'),
         ('R1b', 'R1', 'cái giá của 4-bit + 512 token trên 7B')]
for a, b, why in pairs:
    if a in P and b in P:
        out = f'{OUT_DIR}/paired_{a}_vs_{b}.json'
        ra, rb = f'{EXP_ROOT}/{LABELS[a][1]}', f'{EXP_ROOT}/{LABELS[b][1]}'
        print(f'\n---- {a} vs {b}: {why} ----')
        !python src/eval/compare_runs.py --run_a {ra} --run_b {rb} --label_a {a} --label_b {b} --out {out}
        CMP[f'{a}_vs_{b}'] = json.load(open(out, encoding='utf-8'))
if 'B1_vs_R1b' in CMP:
    c = CMP['B1_vs_R1b']
    verdict = ('7B TỐT HƠN 3B (p < 0,05)' if c['em_diff'] > 0 and c['p_value'] < 0.05 else
               '7B KÉM HƠN 3B (p < 0,05)' if c['em_diff'] < 0 and c['p_value'] < 0.05 else
               'CHƯA PHÂN BIỆT ĐƯỢC (p ≥ 0,05)')
    print(f"\n➜ KẾT LUẬN ablation backbone (cùng 4-bit, 512 token, prompt): {verdict} | ΔEM {100 * c['em_diff']:+.2f} điểm | "
          f"ΔANLS {100 * c['anls_diff']:+.2f} CI95 [{100 * c['anls_ci95'][0]:+.2f}; {100 * c['anls_ci95'][1]:+.2f}]")

## 5. Soi câu trả lời

Độ dài câu trả lời (số từ sau chuẩn hoá) và 6 câu R1 trả lời sai (seed 0). Mục đích: xem lỗi đến từ đọc chữ, từ định dạng (trả lời dài, chen tiếng Anh) hay từ cách chấm.

*Kỳ vọng: nếu R1 trả lời dài hơn gold nhiều, lỗi chủ yếu là định dạng, giống B1. Đó là thứ QLoRA (B2) đã sửa.*

In [ ]:
import random
from src.eval.metrics import exact_match, normalize_vi
nw = lambda s: len(normalize_vi(s).split())
L = {'gold (đáp án đầu)': [nw(p['answers'][0]) for p in P['B1']]}
L.update({k: [nw(p['prediction']) for p in P[k]] for k in P})
display(pd.DataFrame([{'nguồn': k, 'số từ trung bình': sum(v) / len(v), '> 3 từ (%)': 100 * sum(x > 3 for x in v) / len(v)}
                      for k, v in L.items()]).round(2))
if 'R1' in P:
    b1 = {p['question_id']: p for p in P['B1']}
    wrong = [p for p in P['R1'] if not exact_match(p['prediction'], p['answers'])]
    print(f"R1 sai EM {len(wrong)}/{len(P['R1'])} câu")
    for p in random.Random(0).sample(wrong, min(6, len(wrong))):
        print(f"Q: {p['question']}\n   R1  : {p['prediction']!r}\n   B1  : {b1[p['question_id']]['prediction']!r}\n   GOLD: {p['answers'][:3]}")

## 6. Lưu tóm tắt

Ghi `experiments/W07_R1_summary_L4_seed42/r1_summary.json`. Cursor chép file này + các file `paired_*.json` cùng thư mục + `metrics.json`, `predictions.json`, `run_signature.json` (và `oom_fallbacks.json` nếu có) của R1/R1b vào repo. **Không** chép `chunks/`.

*Kỳ vọng: `Ghi -> …/r1_summary.json`. Sau đó **Disconnect and delete runtime**.*

In [ ]:
os.makedirs(OUT_DIR, exist_ok=True)
summary = {
    'what': 'R1: Qwen2-VL-7B-Instruct zero-shot trên ViTextVQA test2000, đối chiếu Table 7 paper ViTextVQA; R1b: cùng model, cấu hình B1 (ablation backbone)',
    'reference': pc.TABLE7_QWENVL7B, 'source': pc.PAPER_SOURCE,
    'pre_registered_rules': {'R1_vs_table7': 'nhất quán nếu |z| <= 1,96 cho cả EM và F1 (tính sai số 100 câu của paper)',
                             'R1b_vs_B1': 'khác biệt nếu McNemar exact p < 0,05 (EM); ANLS kèm CI95 bootstrap ghép cặp'},
    'runs': {k: run for k, (_, run) in LABELS.items()},
    'variants': {k: r['variant'] for k, r in RUNS.items()},
    'table1': rows, 'table7_reports': T7,
    'paired': {k: {kk: vv for kk, vv in c.items() if kk != 'only_b_question_ids'} for k, c in CMP.items()},
    'answer_length_mean_words': {k: sum(v) / len(v) for k, v in L.items()},
    'gpu': GPU,
}
with open(f'{OUT_DIR}/r1_summary.json', 'w', encoding='utf-8') as fp:
    json.dump(summary, fp, ensure_ascii=False, indent=2, default=str)
print('Ghi ->', f'{OUT_DIR}/r1_summary.json')
print('➡ Xong: Runtime -> Disconnect and delete runtime, rồi kiểm tra Manage sessions.')